## Indicators in jarjarquant

This notebook goes over how indicators are implemented in jarjarquant, how they can be used, and the common indicator workflows

---

In [1]:
# Always start with importing the Jarjarquant class and initializing an instance
from jarjarquant import Jarjarquant
import logging

# Set global log level
logging.basicConfig(
    level=logging.INFO,  # Can be INFO, WARNING, ERROR
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

jjq = Jarjarquant()

2025-08-16 20:45:18,940 [INFO] jarjarquant.data_service: Registered equities_metadata view


---
The `list_indicator()` method provides an easy way to look at available indicators

In [2]:
jjq.list_indicators()

[<IndicatorType.ADX: 'adx'>,
 <IndicatorType.ANCHORED_VWAP: 'anchored_vwap'>,
 <IndicatorType.AROON: 'aroon'>,
 <IndicatorType.CHAIKIN_MONEY_FLOW: 'chaikin_money_flow'>,
 <IndicatorType.CMMA: 'cmma'>,
 <IndicatorType.RSI: 'rsi'>,
 <IndicatorType.DETRENDED_RSI: 'detrended_rsi'>,
 <IndicatorType.MACD: 'macd'>,
 <IndicatorType.MOVING_AVERAGE_DIFFERENCE: 'moving_average_difference'>,
 <IndicatorType.PRICE_CHANGE_OSCILLATOR: 'price_change_oscillator'>,
 <IndicatorType.PRICE_INTENSITY: 'price_intensity'>,
 <IndicatorType.REGRESSION_TREND: 'regression_trend'>,
 <IndicatorType.REGRESSION_TREND_DEVIATION: 'regression_trend_deviation'>,
 <IndicatorType.STOCHASTIC: 'stochastic'>,
 <IndicatorType.STOCHASTIC_RSI: 'stochastic_rsi'>]

---
In jarjarquant, each indicator is registered as an `IndicatorType`, which allows for type hints in the editor for a better developer experience. IndicatorType objects can also be used to get details about an indicator using utility method: `get_indicator_parameters()` 

In [3]:
from jarjarquant import IndicatorType, get_indicator_parameters
import json

avwap_params = get_indicator_parameters(IndicatorType.ANCHORED_VWAP)
pi_params = get_indicator_parameters(IndicatorType.PRICE_INTENSITY)
stochrsi_params = get_indicator_parameters(IndicatorType.STOCHASTIC_RSI)
# print(json.dumps(avwap_params, indent=2, default=str))
# print(json.dumps(pi_params, indent=2, default=str))
print(json.dumps(stochrsi_params, indent=2, default=str))


{
  "ohlcv_df": {
    "type": "<class 'polars.dataframe.frame.DataFrame'>",
    "required": true,
    "default": null
  },
  "rsi_period": {
    "type": "<class 'int'>",
    "required": false,
    "default": 14
  },
  "stochastic_period": {
    "type": "<class 'int'>",
    "required": false,
    "default": 14
  },
  "n_smooth": {
    "type": "<class 'int'>",
    "required": false,
    "default": 2
  },
  "transform": {
    "type": "Any",
    "required": false,
    "default": null
  }
}


In [4]:
# Define a spec for the indicator - change any default values if needed
from jarjarquant import IndicatorSpec
avwap_2_21 = IndicatorSpec(IndicatorType.ANCHORED_VWAP, parameters={"threshold_value":0.02, "atr_period":21})
pi_2 = IndicatorSpec(IndicatorType.PRICE_INTENSITY)
stoch_rsi = IndicatorSpec(IndicatorType.STOCHASTIC_RSI)

---
### Indicator Design Evaluation Workflow

A common workflow in early stages of quantitative system development is to evaluate an indicator's design - as opposed to it's performance. A well designed indicator has good statistical properties (stationarity, normality, high entropy) across assets and regimes, which makes it suitable for machine learning algorithms and even rule based systems.

To evaluate an indicator across multiple samples use the `parallel_indicator_distribution_study()` method from `jjq.feature_evaluator`. The workflow looks slightly different based on the data source, but can broadly be broken down into 3 categories based on the type of data source: 1. External API, 2. Custom (Local) Data and 3. Synthetic Data 

#### 1. External API

#### 2. Custom (local) data

In [5]:
from jarjarquant import BarSize, SampleRequest

# Import the SampleRequest class and specify asset_class, date_range, bar_sizes, and any asset_class specific filters
params = None
equity_sample = SampleRequest(sample_type="equities", start_date="2020-02-14", end_date="2024-02-14", bar_size=BarSize.ONE_MINUTE, n_samples = 10, params=params)

print(jjq.feature_evaluator.parallel_indicator_distribution_study(stoch_rsi, equity_sample))

2025-08-16 20:46:36,327 [INFO] jarjarquant.feature_evaluator: Sample request setup time: 0.0000s
2025-08-16 20:46:36,927 [INFO] jarjarquant.feature_evaluator: Data gathering time: 0.5980s
2025-08-16 20:46:36,927 [INFO] jarjarquant.feature_evaluator: Data gathering time: 0.5980s
2025-08-16 20:46:36,952 [INFO] jarjarquant.feature_evaluator: Data preparation time: 0.0250s
2025-08-16 20:46:36,953 [INFO] jarjarquant.feature_evaluator: Processing 8 ticker datasets
2025-08-16 20:46:36,952 [INFO] jarjarquant.feature_evaluator: Data preparation time: 0.0250s
2025-08-16 20:46:36,953 [INFO] jarjarquant.feature_evaluator: Processing 8 ticker datasets
2025-08-16 20:46:40,943 [INFO] jarjarquant.feature_evaluator: Parallel processing time: 3.9892s
2025-08-16 20:46:40,951 [INFO] jarjarquant.data_service: Appended 8 rows to C:\Users\saats\Documents\Trading\jarjarquant\jarjarquant\db\ind_dist_studies_db.parquet
2025-08-16 20:46:40,952 [INFO] jarjarquant.feature_evaluator: Saved 8 rows to ind_dist_studie

{'ADF Test': np.float64(1.0), 'Jarque-Bera Test': np.float64(0.0), 'Relative Entropy': np.float64(2.48), 'Range-IQR Ratio': np.float64(4.74)}
